# TFM: Parte de recopilación de datos.
### Autores: 
- ####  Adrian
- ####  Carlos Mendívil Gómez
- ####  Guillermo
- ####  Jose


## API de Idealista

In [ ]:
import requests
import base64
from datetime import datetime, timedelta

from dotenv import load_dotenv
import os

import pandas as pd

In [ ]:
#------------------------------------------------------------------------------------------------------------
# Pedir a la API que me devuelva un token 
def idealista_token(api_key, secret):
    """ Request a token from OAuth authentification

    Parameters
    ----------
    api_key(str): Api key
    secret(str): secreto

    Returns
    -------
    access_token, fecha caducidad
    """
    real_credentials = api_key+":"+secret
    base64_bytes = base64.b64encode(real_credentials.encode("ascii"))
    credentials = base64_bytes.decode("ascii")


    req = requests.post(
        "https://api.idealista.com/oauth/token",
        "grant_type=client_credentials&scope=read",
        headers={
            "Authorization": f"Basic {credentials}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
    ).json()
    return req["access_token"], datetime.now() + timedelta(seconds=req["expires_in"])

#--------------------------------------------------------------------------------------------------------------
# Realiza una consulta a la API de idealista
def query(token, request):
    """Realiza una consulta a la API de idealista
    Parameter
    ---------
    token: token recibido por la API de outhenticator
    request: diccionario con los datos de la solicitud

    Returns
    -------
    Devuelve la respuesta de la API
    """

    response = requests.post(
            "https://api.idealista.com/3.5/es/search",
            headers={
                "Authorization": f"Bearer {token}",
                "User-Agent": "curl/8.3.0"
            },
            data=request,
        )
    return response.json()


In [ ]:
load_dotenv()
api_key = os.getenv("api_key")
secret = os.getenv("secret")

In [ ]:
for num_page in range(1,100):
    filter_request = {
        "country": "es",
        "operation": "sale",
        "propertyType": "homes",
        "locationId": "0-EU-ES-28",
        "maxItems": 50,
        "numPage": num_page
    }
    token, expires = idealista_token(api_key, secret)
    print(token)
    response = query(token, filter_request)

    # Lista de viviendas dentro del JSON
    viviendas = response["elementList"]

    # Usamos json_normalize con sep='.' para que las claves anidadas se aplanen con ese separador
    df = pd.json_normalize(viviendas, sep='.')

    df.to_csv(f"datos/datos_sec_pag_{num_page}.csv", index=False)

## Combinación de todos los csv

In [1]:
import numpy as np
import pandas as pd

import re
import requests
from io import StringIO

In [2]:
# URLs base
repo_html_url = "https://github.com/guille1006/TFM/tree/main/Idealista/datos"
raw_base = "https://raw.githubusercontent.com/guille1006/TFM/main/Idealista/datos/"

# Obtener lista de archivos
html = requests.get(repo_html_url).text
csv_files = re.findall(r'datos_pag_\d+\.csv', html)
csv_files = [(int(file.split('_')[2].split('.')[0]), file) for file in set(csv_files)]
csv_files = sorted(csv_files, key=lambda x: x[0])

print(f"Archivos encontrados: {len(csv_files)}")


# Al haber automatizado la obtención de datos, hay algunos .csv que están vacíos
dataframes = dict()
errores = []

for num_page, file in csv_files:
    file_url = raw_base + file

    # Descargaremos toda la información dentro de cada enlace para ver que tiene
    resp = requests.get(file_url)
    content = resp.text.strip()

    # Puede ser que no tengan contenido debido a que se acabaron el tipo de viviendas para el filtro usado
    if not content:
        errores.append((num_page, "Archivo vacío"))
        continue

    # Usaremos StringIO para poder pasar el contenido a un dataframe de pandas
    df = pd.read_csv(StringIO(content))

    # Por motivos de eficiencia de tiempo, hemos decidido ir guardando todos los df en una lista
    # que luego usaremos para concatenar todos ellos
    dataframes[num_page] = df

# Vamos a ordenar el set de all_columns y guardarlo como una lista
dfs = list(dataframes.values())

raw_data = pd.concat(dfs, ignore_index=True, sort=True)

# Tambien eliminaremos las filas duplicadas
initial_rows = raw_data.shape[0]
raw_data = raw_data.drop_duplicates()
final_rows = raw_data.shape[0]

print(f"Teniamos un total de {final_rows-initial_rows} duplicadas")

raw_data.to_csv("data/raw_data.csv", index=False)

Archivos encontrados: 498
Teniamos un total de -4210 duplicadas
